# Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta 
from zoneinfo import ZoneInfo
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

In [ ]:
SERIES_IDS = ['UNRATE', 'INDPRO', 'JTSJOL']

## Import raw feature data from the db

In [ ]:
data = database.db_requester_fetchall(sql_script="all_obs_FREDObservations.sql")

In [ ]:
arr_raw = np.array(data)
arr_raw[np.where(arr_raw == None)[0]]
# np.isnan(arr_raw)

## Data prep
- Impute missing values

In [ ]:
df_raw = pd.DataFrame(data, columns=['series_id', 'year_month', 'value', 'timestamp'])
df_raw['year_month'] = pd.to_datetime(df_raw['year_month'])
df_raw

In [ ]:
df = df_raw.pivot(
    index='year_month',
    columns='series_id',
    values='value'
).reset_index()
df['year_month'] = df['year_month'].dt.to_period('M')
df.columns.name = None

df

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
df_imputed = df.copy()
df_imputed[SERIES_IDS] = df[SERIES_IDS].interpolate(method="linear")
df_imputed[df_imputed.isna().any(axis=1)]
df_imputed

## Create a features df
If the most recent available `UNRATE` if of time $t$, then the values of `INDPRO` and `JTSJOL` are of time $t-1$

In [ ]:
df_features = pd.DataFrame({
    "year_month_t": df_imputed['year_month'].copy(),
    "log_10_UNRATE_t": df_imputed['UNRATE'].apply(np.log10).copy(),
    "log_10_INDPRO_t-1": df_imputed['INDPRO'].shift(1).apply(np.log10).copy(),
    "log_10_JTSJOL_t-1": df_imputed['JTSJOL'].shift(1).apply(np.log10).copy(),
    "UNRATE_t": df_imputed['UNRATE'].copy(),
    "INDPRO_t-1": df_imputed['INDPRO'].shift(1).copy(),
    "JTSJOL_t-1": df_imputed['JTSJOL'].shift(1).copy(),
    "UNRATE_t+1": df_imputed['UNRATE'].shift(-1).copy(),
})

df_features


In [ ]:
features = df_features.iloc[:, 1:-1].columns.to_list()
target = "UNRATE_t+1"

In [ ]:
cols_set = df_features.iloc[:, 1:-1].columns.to_list()

ncols = 3
nrows = int(np.ceil(len(cols_set) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 3.5 * nrows),
    sharex=True
)

# Makes this work whether you have 1 row or multiple rows
axes = np.array(axes).flatten()

for ax, col in zip(axes, cols_set):
    sns.lineplot(
        data=df_features,
        x=df_features['year_month_t'].dt.to_timestamp(),
        y=col,
        ax=ax
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("")

for ax in axes[len(cols_set):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

**Changes set is more stable**

In [ ]:
ncols = 3
nrows = int(np.ceil(len(features) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 4 * nrows)
)

axes = np.array(axes).flatten()

for ax, feature in zip(axes, features):
    sns.scatterplot(
        data=df_features,
        x=feature,
        y=target,
        ax=ax,
        alpha=0.6
    )

    ax.set_title(f"{feature} → {target}")
    ax.set_xlabel(feature)
    ax.set_ylabel(target)


for ax in axes[len(features):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(
    df_features[features + [target]],
    diag_kind="hist",
    corner=True
)

plt.show()

In [ ]:
df_features[features + [target]].corr()

In [ ]:
df_features[df_features.isna().any(axis=1)]

# Export the features and target as a df
Implement a fail-safe to ensure that the last month included has complete features or is excluded ('log_10_UNRATE_t', 'log_10_INDPRO_t-1','log_10_JTSJOL_t-1', 'UNRATE_t', 'INDPRO_t-1', 'JTSJOL_t-1')

In [ ]:
# # manipulate df with missing values for testing
# df_features.loc[df_features.index[-1], ['log_10_UNRATE_t', 'UNRATE_t']] = np.nan
# df_features.loc[df_features.index[-2], ['UNRATE_t+1']] = np.nan
# df_features

In [ ]:
# get the indexes of row with NaN values, excluding the target column UNRATE_t+1 which naturally has a NaN in its last row.
nan_rows = df_features[df_features.iloc[:,1:-1].isna().any(axis=1)].index.to_list()
df_features = df_features.drop(nan_rows, axis=0).reset_index(drop=True)
df_features

In [ ]:
import feature_pipeline

feature_pipeline.get_features()